# 📖 TasmiqAI — Quran Expert System

This notebook integrates the **complete 30-Juz Quran JSON dataset** with the **Wav2Vec2 Quran Phonetics engine** to build an intelligent expert system for Tahfiz recitation assessment.

**Dataset path:** `C:\Users\nabil\.gemini\antigravity\scratch\quranjson\source`

### What this notebook does:
1. 📂 Load and explore the full 114-surah / 30-juz dataset
2. 🧠 Build an expert system that maps recitation to Quran text (ayah matching)
3. 🎙️ Run phonetic recognition on a recorded audio file
4. ✅ Compare detected phonetics against the expected ayah text
5. 📊 Generate a simple accuracy/feedback report

---
## Cell 1 — Install Required Dependencies
Run this cell **once** to install all packages into the current Anaconda environment.

In [2]:
import sys
!{sys.executable} -m pip install soundfile librosa transformers torch --quiet
print("✅ Dependencies ready.")

✅ Dependencies ready.


---
## Cell 2 — Load the Quran JSON Dataset

In [3]:
import json
import os
from pathlib import Path

# ============================================================
#  DATASET PATH — points to the 30-juz Quran JSON collection
# ============================================================
DATASET_ROOT = Path(r"C:\Users\nabil\.gemini\antigravity\scratch\quranjson\source")
SURAH_DIR    = DATASET_ROOT / "surah"
JUZ_FILE     = DATASET_ROOT / "juz.json"
SURAH_INDEX  = DATASET_ROOT / "surah.json"

# ── Load juz map ────────────────────────────────────────────
with open(JUZ_FILE, "r", encoding="utf-8") as f:
    juz_data = json.load(f)

# ── Load surah index ─────────────────────────────────────────
with open(SURAH_INDEX, "r", encoding="utf-8") as f:
    surah_index = json.load(f)   # list of {index, name, ...}

# ── Load all 114 surahs into a dict {int -> surah_data} ─────
quran = {}
surah_files = sorted(SURAH_DIR.glob("surah_*.json"),
                     key=lambda p: int(p.stem.split("_")[1]))

for sf_path in surah_files:
    with open(sf_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    surah_num = int(data["index"])
    quran[surah_num] = data

print(f"✅  Loaded {len(quran)} surahs and {len(juz_data)} juz from dataset.")
print(f"📂  Dataset root: {DATASET_ROOT}")

✅  Loaded 114 surahs and 30 juz from dataset.
📂  Dataset root: C:\Users\nabil\.gemini\antigravity\scratch\quranjson\source


---
## Cell 3 — Explore Dataset Structure

In [4]:
# ── Quick statistics ─────────────────────────────────────────
total_verses = sum(s["count"] for s in quran.values())
print(f"📖  Total surahs  : {len(quran)}")
print(f"📜  Total verses  : {total_verses}")
print(f"📚  Total juz     : {len(juz_data)}")
print()

# ── Print first 5 surahs ─────────────────────────────────────
print("First 5 surahs:")
for i in range(1, 6):
    s = quran[i]
    print(f"  Surah {i:>3} — {s['name']:<20} ({s['count']} verses)")

print()
# ── Sample: Al-Fatiha ────────────────────────────────────────
print("Sample — Surah 1 (Al-Fatiha):")
for k, v in quran[1]["verse"].items():
    print(f"  {k}: {v}")

📖  Total surahs  : 114
📜  Total verses  : 6236
📚  Total juz     : 30

First 5 surahs:
  Surah   1 — al-Fatihah           (7 verses)
  Surah   2 — al-Baqarah           (286 verses)
  Surah   3 — Ali Imran            (200 verses)
  Surah   4 — an-Nisa'             (176 verses)
  Surah   5 — al-Mai'dah           (120 verses)

Sample — Surah 1 (Al-Fatiha):
  verse_1: ﻿بِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ
  verse_2: ٱلْحَمْدُ لِلَّهِ رَبِّ ٱلْعَٰلَمِينَ
  verse_3: ٱلرَّحْمَٰنِ ٱلرَّحِيمِ
  verse_4: مَٰلِكِ يَوْمِ ٱلدِّينِ
  verse_5: إِيَّاكَ نَعْبُدُ وَإِيَّاكَ نَسْتَعِينُ
  verse_6: ٱهْدِنَا ٱلصِّرَٰطَ ٱلْمُسْتَقِيمَ
  verse_7: صِرَٰطَ ٱلَّذِينَ أَنْعَمْتَ عَلَيْهِمْ غَيْرِ ٱلْمَغْضُوبِ عَلَيْهِمْ وَلَا ٱلضَّآلِّينَ


---
## Cell 4 — Expert System Helper Functions

These functions let you look up any ayah, search by juz/surah, and prepare ground-truth text for comparison.

In [5]:
# ============================================================
#  EXPERT SYSTEM — Quran Knowledge Base Functions
# ============================================================

def get_ayah(surah_num: int, ayah_num: int) -> str:
    """Return the Arabic text of a specific ayah."""
    surah = quran.get(surah_num)
    if surah is None:
        return f"[Surah {surah_num} not found]"
    key = f"verse_{ayah_num}"
    return surah["verse"].get(key, f"[Ayah {ayah_num} not found in Surah {surah_num}]")


def get_surah_verses(surah_num: int) -> dict:
    """Return all verses of a surah as a dict {verse_N: text}."""
    surah = quran.get(surah_num, {})
    return surah.get("verse", {})


def get_juz_surahs(juz_num: int) -> list:
    """Return surah/verse range info for a given juz (1–30)."""
    for j in juz_data:
        if int(j["index"]) == juz_num:
            return j
    return {}


def get_surah_name(surah_num: int) -> str:
    """Return the name of a surah."""
    return quran.get(surah_num, {}).get("name", "Unknown")


def list_surahs_in_juz(juz_num: int):
    """Print a summary of which surahs are in a given juz."""
    juz = get_juz_surahs(juz_num)
    if not juz:
        print(f"Juz {juz_num} not found.")
        return
    start_surah = int(juz["start"]["index"])
    end_surah   = int(juz["end"]["index"])
    start_verse  = juz["start"]["verse"]
    end_verse    = juz["end"]["verse"]
    print(f"\n📗 Juz {juz_num:02d}")
    print(f"  Starts: Surah {start_surah} ({get_surah_name(start_surah)}) — {start_verse}")
    print(f"  Ends  : Surah {end_surah} ({get_surah_name(end_surah)}) — {end_verse}")
    print(f"  Surahs: {', '.join(str(n) for n in range(start_surah, end_surah+1))}")


# ── Quick demo ───────────────────────────────────────────────
print("Ayah 1 of Al-Fatiha:", get_ayah(1, 1))
print("Ayah 255 of Al-Baqara (Ayat Al-Kursi):", get_ayah(2, 255))
list_surahs_in_juz(30)
print("\n✅ Expert system knowledge base ready.")

Ayah 1 of Al-Fatiha: ﻿بِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ
Ayah 255 of Al-Baqara (Ayat Al-Kursi): ٱللَّهُ لَآ إِلَٰهَ إِلَّا هُوَ ٱلْحَىُّ ٱلْقَيُّومُ ۚ لَا تَأْخُذُهُۥ سِنَةٌۭ وَلَا نَوْمٌۭ ۚ لَّهُۥ مَا فِى ٱلسَّمَٰوَٰتِ وَمَا فِى ٱلْأَرْضِ ۗ مَن ذَا ٱلَّذِى يَشْفَعُ عِندَهُۥٓ إِلَّا بِإِذْنِهِۦ ۚ يَعْلَمُ مَا بَيْنَ أَيْدِيهِمْ وَمَا خَلْفَهُمْ ۖ وَلَا يُحِيطُونَ بِشَىْءٍۢ مِّنْ عِلْمِهِۦٓ إِلَّا بِمَا شَآءَ ۚ وَسِعَ كُرْسِيُّهُ ٱلسَّمَٰوَٰتِ وَٱلْأَرْضَ ۖ وَلَا يَـُٔودُهُۥ حِفْظُهُمَا ۚ وَهُوَ ٱلْعَلِىُّ ٱلْعَظِيمُ

📗 Juz 30
  Starts: Surah 78 (an-Naba') — verse_1
  Ends  : Surah 114 (an-Nas) — verse_6
  Surahs: 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114

✅ Expert system knowledge base ready.


---
## Cell 5 — Load Wav2Vec2 Phonetics Model

> ⚠️ This will **download ~400 MB** the first time. After that it is cached locally.

In [ ]:
import torch
import logging
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

logging.basicConfig(level=logging.WARNING)   # suppress verbose HF logs

MODEL_NAME = "TBOGamer22/wav2vec2-quran-phonetics"
print(f"⏳ Loading model '{MODEL_NAME}'... (first run may take a few minutes)")

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model     = Wav2Vec2ForCTC.from_pretrained(MODEL_NAME)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = model.to(device)

print(f"✅ Model loaded on device: {device}")

⏳ Loading model 'TBOGamer22/wav2vec2-quran-phonetics'... (first run may take a few minutes)


---
## Cell 6 — Phonetic Recognition Engine

Transcribes an audio file using the Wav2Vec2 model.

In [8]:
import soundfile as sf
import librosa
import numpy as np

def transcribe_audio(audio_path: str) -> str:
    """
    Load an audio file, resample to 16 kHz, and run Wav2Vec2 phonetic recognition.
    Returns the detected phonetic string.
    """
    if not os.path.exists(audio_path):
        raise FileNotFoundError(
            f"Audio file not found: {audio_path}\n"
            "👉 Please record your recitation and save it as a .wav file, "
            "then update AUDIO_FILE below."
        )

    print(f"🎙️  Loading audio: {audio_path}")
    audio_array, sr = sf.read(audio_path)

    # Stereo → Mono
    if audio_array.ndim > 1:
        audio_array = audio_array.mean(axis=1)
        print("   (converted stereo → mono)")

    # Resample to 16 kHz
    if sr != 16000:
        print(f"   (resampling {sr} Hz → 16000 Hz)")
        audio_array = librosa.resample(y=audio_array.astype(np.float32),
                                       orig_sr=sr, target_sr=16000)

    print("🤖  Running phonetic inference...")
    inputs = processor(audio_array, sampling_rate=16000,
                       return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        logits = model(**inputs).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    phonetics = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return phonetics


print("✅  transcribe_audio() function defined.")

✅  transcribe_audio() function defined.


---
## Cell 7 — 🎯 Run Assessment (Dataset Audio)

This cell is **fully self-contained** — run it alone or after other cells.
Set `TARGET_SURAH` and `TARGET_AYAH`; the audio is pulled automatically from the dataset MP3 files.

In [1]:
# ============================================================
#  ⚙️  CONFIGURE YOUR TEST — change these two values only
# ============================================================
TARGET_SURAH = 1   # 1–114
TARGET_AYAH  = 1   # verse number inside that surah
# ============================================================

import json, os, logging
import numpy as np
from pathlib import Path
import librosa
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

logging.basicConfig(level=logging.WARNING)

DATASET_ROOT = Path(r"C:\Users\nabil\.gemini\antigravity\scratch\quranjson\source")
AUDIO_ROOT   = DATASET_ROOT / "audio"
SURAH_DIR    = DATASET_ROOT / "surah"

# ── 1. Load dataset (if not already in memory) ───────────────
_quran = {}
for _sf in sorted(SURAH_DIR.glob("surah_*.json"),
                  key=lambda p: int(p.stem.split('_')[1])):
    with open(_sf, 'r', encoding='utf-8') as _f:
        _d = json.load(_f)
    _quran[int(_d['index'])] = _d
print(f"✅  Dataset loaded: {len(_quran)} surahs.")

def _ayah(s, a): return _quran.get(s, {}).get('verse', {}).get(f'verse_{a}', '[not found]')
def _name(s):    return _quran.get(s, {}).get('name', 'Unknown')

# ── 2. Resolve audio path from dataset ───────────────────────
audio_path    = AUDIO_ROOT / f"{TARGET_SURAH:03d}" / f"{TARGET_AYAH:03d}.mp3"
expected_text = _ayah(TARGET_SURAH, TARGET_AYAH)
surah_name    = _name(TARGET_SURAH)

print("=" * 62)
print(f"📖  Target      : Surah {TARGET_SURAH} ({surah_name}), Ayah {TARGET_AYAH}")
print(f"📝  Arabic text : {expected_text}")
print(f"🔊  Audio file  : {audio_path}")
print("=" * 62)

if not audio_path.exists():
    print(f"\n⚠️  Audio file not found: {audio_path}")
    print("   Check that TARGET_SURAH and TARGET_AYAH are valid.")
else:
    # ── 3. Load model ─────────────────────────────────────────
    _MODEL = "TBOGamer22/wav2vec2-quran-phonetics"
    print("\n⏳ Loading Wav2Vec2 model (cached after first run)...")
    _processor = Wav2Vec2Processor.from_pretrained(_MODEL)
    _model     = Wav2Vec2ForCTC.from_pretrained(_MODEL)
    _model.eval()
    _device = "cuda" if torch.cuda.is_available() else "cpu"
    _model  = _model.to(_device)
    print(f"✅ Model ready on: {_device}")

    # ── 4. Load MP3 audio via librosa ────────────────────────
    print(f"\n🎙️  Loading audio: {audio_path.name}")
    _audio, _sr = librosa.load(str(audio_path), sr=16000, mono=True)
    print(f"   Duration: {len(_audio)/16000:.2f}s  |  16000 Hz")

    # ── 5. Phonetic inference ─────────────────────────────────
    print("🤖  Running phonetic inference...")
    _inputs = _processor(_audio, sampling_rate=16000, return_tensors="pt", padding=True)
    _inputs = {k: v.to(_device) for k, v in _inputs.items()}
    with torch.inference_mode():
        _logits = _model(**_inputs).logits
    _ids       = torch.argmax(_logits, dim=-1)
    _phonetics = _processor.batch_decode(_ids, skip_special_tokens=True)[0]

    # ── 6. Result ─────────────────────────────────────────────
    print()
    print("=" * 62)
    print("📊  ASSESSMENT RESULT")
    print("=" * 62)
    print(f"📝  Arabic (expected)  : {expected_text}")
    print(f"🎙️  Phonetics detected : {_phonetics}")
    print("=" * 62)
    print("✅  Assessment complete!")


✅  Dataset loaded: 114 surahs.
📖  Target      : Surah 1 (al-Fatihah), Ayah 1
📝  Arabic text : ﻿بِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ
🔊  Audio file  : C:\Users\nabil\.gemini\antigravity\scratch\quranjson\source\audio\001\001.mp3

⏳ Loading Wav2Vec2 model (cached after first run)...


Loading weights:   0%|          | 0/213 [00:00<?, ?it/s]

✅ Model ready on: cpu

🎙️  Loading audio: 001.mp3
   Duration: 5.46s  |  16000 Hz
🤖  Running phonetic inference...

📊  ASSESSMENT RESULT
📝  Arabic (expected)  : ﻿بِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ
🎙️  Phonetics detected : bils-'milāliranānrhīmi
✅  Assessment complete!


---
## Cell 8 — 📚 Browse Any Juz / Surah / Ayah

Use this cell to interactively explore any part of the Quran dataset.

In [ ]:
# ── List all content of a specific Juz ──────────────────────
JUZ_TO_BROWSE = 30           # Change to any juz 1–30
list_surahs_in_juz(JUZ_TO_BROWSE)

print()

# ── Print all ayahs of a specific surah ─────────────────────
SURAH_TO_BROWSE = 114        # Change to any surah 1–114
verses = get_surah_verses(SURAH_TO_BROWSE)
print(f"\n📖 Surah {SURAH_TO_BROWSE} — {get_surah_name(SURAH_TO_BROWSE)}")
for verse_key, text in verses.items():
    num = verse_key.replace("verse_", "")
    print(f"  Ayah {num:>4}: {text}")

---
## Cell 9 — 📊 Dataset Statistics by Juz

In [ ]:
print(f"{'Juz':>5} | {'Start':^30} | {'End':^30}")
print("-" * 72)
for j in juz_data:
    juz_num     = int(j["index"])
    start_name  = j["start"]["name"]
    start_verse = j["start"]["verse"]
    end_name    = j["end"]["name"]
    end_verse   = j["end"]["verse"]
    print(f"{juz_num:>5} | {start_name+' '+start_verse:^30} | {end_name+' '+end_verse:^30}")